# SMS Spam Detection: Legitimate Promotion-Word Messages vs Genuine Spam

**NLP Course Project, ARTI501 (9MA1), Group 6, Option 1: Spam Detection**
Team: Ali Fawzi Alshehri, Osama Saeed Al Haddaf, Hussain Ali Alabdurabalnabi

**Task.** The notebook accepts an SMS message and returns **Spam** or **Not spam**.

**Experimental question (approved).** Which method distinguishes legitimate promotion-word SMS messages
from genuine spam more reliably: TF-IDF word features with Logistic Regression, or pretrained sentence
representations with a classifier?

**How to run.** Built for **Google Colab**: open the notebook and choose *Runtime → Run all*. No GPU is needed.
It also runs in local Jupyter after `pip install -r requirements.txt`. All paths are relative. The first run
downloads the dataset (about 200 KB) and the pretrained MiniLM model (about 90 MB).

**Sections:** 1 Setup · 2 Data · 3 Method 1 · 4 Method 2 · 5 Results · 6 Try it · 7 Sources

## 1. Setup

* **1.1** installs the one package Colab may be missing.
* **1.2** imports everything, fixes one random seed, sets relative folder paths and prints the package versions
  used, which are part of the reproducibility record.
* **1.3** defines a helper that saves each results table and says whether a re-run reproduced it.

In [1]:
# 1.1  sentence-transformers (Method 2) is normally preinstalled on Colab; install it only if it is missing
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
print("sentence-transformers is available")

sentence-transformers is available


In [2]:
# 1.2  Imports, seed, paths, versions
import hashlib
import html
import os
import platform
import re
import sys
import urllib.request
import warnings
import zipfile
from pathlib import Path

# Harmless environment warnings: tqdm cannot draw progress bars without ipywidgets, and older scikit-learn
# paired with newer SciPy reports an unused solver option
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message="Unknown solver options")

# Quieter model download: no progress bars or Windows symlink warnings
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import scipy
import sentence_transformers
import sklearn
import torch
from huggingface_hub.utils import logging as hub_logging
from scipy.stats import binomtest
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.pipeline import Pipeline
from transformers.utils import logging as hf_logging

hub_logging.set_verbosity_error()
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

SEED = 42                      # used for every random step in this notebook
np.random.seed(SEED)
torch.manual_seed(SEED)

IN_COLAB = "google.colab" in sys.modules

# Relative paths: the same code works on Colab (/content) and in a local copy of the repository
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 120)

print("Environment :", "Google Colab" if IN_COLAB else "local Jupyter")
print("Python      :", platform.python_version())
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("scipy       :", scipy.__version__)
print("torch       :", torch.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

Environment : Google Colab
Python      : 3.13.18
pandas      : 2.2.3
numpy       : 2.4.6
scikit-learn: 1.6.1
scipy       : 1.16.3
torch       : 2.14.1+cpu
sentence-transformers: 5.7.0


In [3]:
# 1.3  Save results tables and check them against the previously saved copy
def save_result(table, name):
    """Write a table to results/<name>.csv and report whether it matches the copy saved by an earlier run."""
    path = RESULTS_DIR / f"{name}.csv"
    new_csv = table.to_csv(lineterminator="\n", float_format="%.4f")
    if not path.exists():
        status = "saved"
    elif path.read_text(encoding="utf-8") == new_csv:
        status = "reproduced: identical to the saved copy"
    else:
        status = "UPDATED: differs from the saved copy"
    path.write_text(new_csv, encoding="utf-8")
    print(f"{path.as_posix()}: {status}")

## 2. Data

**Dataset.** UCI SMS Spam Collection v.1 (Almeida & Gómez Hidalgo, 2011), a public corpus of English SMS
messages labelled *ham* (legitimate) or *spam*. Sources: the UK Grumbletext forum (spam), the NUS SMS Corpus
(Singapore, ham), Caroline Tagg's PhD thesis (ham) and the SMS Spam Corpus v.0.1 Big.
The corpus readme allows free use with citation; copyright stays with the authors (see Section 7).

### 2.1 Get the dataset
The notebook uses `data/smsspamcollection.zip` if it is present, otherwise it downloads the file from UCI.
A SHA-256 checksum confirms that the file is exactly the one our results are based on.

In [4]:
DATA_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
BACKUP_DATA_URL = None   # TODO: raw GitHub URL of data/smsspamcollection.zip once the repository is public
ZIP_PATH = DATA_DIR / "smsspamcollection.zip"
EXPECTED_SHA256 = "1587ea43e58e82b14ff1f5425c88e17f8496bfcdb67a583dbff9eefaf9963ce3"


def sha256_of(path):
    """Return the SHA-256 digest of a file."""
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def ensure_dataset():
    """Reuse data/smsspamcollection.zip if present; otherwise download it, then verify the checksum."""
    if not ZIP_PATH.exists():
        for url in (DATA_URL, BACKUP_DATA_URL):
            if url is None:
                continue
            try:
                print("Downloading", url)
                part = ZIP_PATH.with_suffix(".part")
                urllib.request.urlretrieve(url, part)
                part.replace(ZIP_PATH)          # only keep complete downloads
                break
            except Exception as err:
                print("  download failed:", err)
    if not ZIP_PATH.exists():
        raise FileNotFoundError(
            "Dataset missing. Download it from https://archive.ics.uci.edu/dataset/228/sms+spam+collection "
            "and save it as data/smsspamcollection.zip")
    digest = sha256_of(ZIP_PATH)
    assert digest == EXPECTED_SHA256, f"Unexpected dataset file (sha256 {digest}); results may differ from the report."
    print("Dataset ready:", ZIP_PATH, "(checksum OK)")


ensure_dataset()

Dataset ready: data/smsspamcollection.zip (checksum OK)


### 2.2 Load the raw messages
Each line of the file is `label<TAB>text`. We split each line on its first tab ourselves rather than using
`pd.read_csv`, because some messages contain double quotes that a CSV parser would misread.
The line number becomes a stable message ID (`msg_id`).

In [5]:
LABEL_NAMES = {"ham": "Not spam", "spam": "Spam"}


def load_raw_messages(zip_path):
    """Read the UCI file into a DataFrame with columns msg_id, label_raw, text_raw."""
    with zipfile.ZipFile(zip_path) as zf:
        raw_text = zf.read("SMSSpamCollection").decode("utf-8")
    rows = [line.split("\t", 1) for line in raw_text.splitlines()]
    frame = pd.DataFrame(rows, columns=["label_raw", "text_raw"])
    frame.insert(0, "msg_id", range(len(frame)))
    return frame


raw_df = load_raw_messages(ZIP_PATH)
assert len(raw_df) == 5574 and set(raw_df["label_raw"]) == {"ham", "spam"}
print(f"{len(raw_df):,} messages loaded")
raw_df.head()

5,574 messages loaded


,msg_id,label_raw,text_raw
0,0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,1,ham,Ok lar... Joking wif u oni...
2,2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std t...
3,3,ham,U dun say so early hor... U c already then say...
4,4,ham,"Nah I don't think he goes to usf, he lives around here though"


### 2.3 Clean the text
Light cleaning, applied identically to every message and to both methods:

1. **Decode HTML entities.** The file stores `<` as `&lt;` and `&` as `&amp;`, for example `&lt;#&gt;` becomes `<#>`.
2. **Repair Windows-1252 leftovers.** A few apostrophes and dashes were stored as control codes (`\x92` becomes `’`).
3. **Collapse repeated whitespace** and trim.

We deliberately **keep** capital letters, punctuation, digits and currency symbols. Signals such as `URGENT!`,
`£1000` and phone numbers are typical of spam, and the pretrained model in Method 2 was trained on natural
text. Lowercasing for TF-IDF happens inside the Method 1 pipeline, not here. We do no stop-word removal or
stemming: messages are short, and SMS spelling (`ur`, `2nite`) defeats standard stemmers.

In [6]:
CP1252_CONTROL = re.compile("[\x80-\x9f]")


def clean_text(text):
    """Label-independent cleaning shared by both methods."""
    text = html.unescape(text)                                       # &lt;#&gt; -> <#>, &amp; -> &
    text = CP1252_CONTROL.sub(                                       # \x92 -> ’
        lambda m: m.group().encode("latin-1").decode("cp1252", errors="ignore"), text)
    return re.sub(r"\s+", " ", text).strip()                         # collapse whitespace


df = raw_df.copy()
df["text"] = df["text_raw"].map(clean_text)
df["label"] = df["label_raw"].map(LABEL_NAMES)
df["y"] = (df["label"] == "Spam").astype(int)                        # 1 = Spam, 0 = Not spam

changed = df["text"] != df["text_raw"]
print(f"Cleaning changed {changed.sum()} of {len(df):,} messages; empty after cleaning: {(df['text'] == '').sum()}")
df.loc[changed, ["text_raw", "text"]].head(4)

Cleaning changed 553 of 5,574 messages; empty after cleaning: 0


,text_raw,text
18,Fine if thats the way u feel. Thats the way its gota b,Fine if that’s the way u feel. That’s the way its gota b
28,"I'm back &amp; we're packing the car now, I'll let you know if there's room","I'm back & we're packing the car now, I'll let you know if there's room"
31,Yeah he got in at 2 and was v apologetic. n had fallen out and she was actin like spoilt child and he got caught up ...,Yeah he got in at 2 and was v apologetic. n had fallen out and she was actin like spoilt child and he got caught up ...
44,Great! I hope you like your man well endowed. I am &lt;#&gt; inches...,Great! I hope you like your man well endowed. I am <#> inches...


### 2.4 Remove exact duplicates
The brief requires duplicates to stay in one split. We go one step further and keep only the **first copy** of
every message that appears more than once (ignoring case and spacing), so each unique message counts once.
First we check that no repeated text carries two different labels.

In [7]:
df["dup_key"] = df["text"].str.lower()
assert (df.groupby("dup_key")["label"].nunique() == 1).all(), "A repeated text has conflicting labels"

n_before = len(df)
df = df.drop_duplicates("dup_key", keep="first").reset_index(drop=True)
print(f"Removed {n_before - len(df)} exact duplicates: {n_before:,} -> {len(df):,} unique messages")

class_counts = pd.DataFrame({
    "Original file": raw_df["label_raw"].map(LABEL_NAMES).value_counts(),
    "After de-duplication": df["label"].value_counts(),
}).loc[["Not spam", "Spam"]]
class_counts.loc["Total"] = class_counts.sum()
save_result(class_counts, "class_counts")
print(f"Spam share after de-duplication: {df['y'].mean():.1%}")
class_counts

Removed 416 exact duplicates: 5,574 -> 5,158 unique messages
results/class_counts.csv: saved
Spam share after de-duplication: 12.4%


,Original file,After de-duplication
Not spam,4827,4517
Spam,747,641
Total,5574,5158


### 2.5 Group near-duplicates
Spam campaigns often send the same template with a different phone number or prize code. If one copy were in
training and another in test, the test score would be inflated. We give each message a **template key**
(lowercase, digits and punctuation removed). The split in 2.7 assigns whole groups, never individual
messages, so near-copies always land in the same split.

In [8]:
def template_key(text):
    """Key that ignores case, digits and punctuation, so templated near-copies share one key."""
    key = re.sub(r"\d+", " ", text.lower())       # phone numbers, codes, prices
    key = re.sub(r"[^a-z£ ]", " ", key)            # punctuation and other symbols
    key = re.sub(r"\s+", " ", key).strip()
    return key or text.lower()                     # a message with no letters keeps its own key


df["group_key"] = df["text"].map(template_key)
assert (df.groupby("group_key")["label"].nunique() == 1).all(), "A template group mixes labels"

group_sizes = df.groupby("group_key").size()
multi = group_sizes[group_sizes > 1]
print(f"{len(group_sizes):,} template groups; {len(multi)} groups hold near-duplicates ({multi.sum()} messages)")

# Illustration: the largest group of near-duplicates
df.loc[df["group_key"] == group_sizes.idxmax(), ["msg_id", "label", "text"]]

5,091 template groups; 50 groups hold near-duplicates (117 messages)


,msg_id,label,text
522,531,Spam,PRIVATE! Your 2003 Account Statement for 07815296484 shows 800 un-redeemed S.I.M. points. Call 08718738001 Identifie...
582,593,Spam,PRIVATE! Your 2003 Account Statement for 07753741225 shows 800 un-redeemed S. I. M. points. Call 08715203677 Identif...
637,648,Spam,PRIVATE! Your 2003 Account Statement for shows 800 un-redeemed S. I. M. points. Call 08715203694 Identifier Code: 40...
1541,1594,Spam,PRIVATE! Your 2003 Account Statement for shows 800 un-redeemed S. I. M. points. Call 08719899230 Identifier Code: 41...
3834,4088,Spam,PRIVATE! Your 2003 Account Statement for 07973788240 shows 800 un-redeemed S. I. M. points. Call 08715203649 Identif...
4343,4658,Spam,PRIVATE! Your 2003 Account Statement for shows 800 un-redeemed S. I. M. points. Call 08718738002 Identifier Code: 48...
4511,4841,Spam,PRIVATE! Your 2003 Account Statement for shows 800 un-redeemed S. I. M. points. Call 08715203652 Identifier Code: 42...
5093,5503,Spam,PRIVATE! Your 2003 Account Statement for 07808247860 shows 800 un-redeemed S. I. M. points. Call 08719899229 Identif...
5116,5528,Spam,PRIVATE! Your 2003 Account Statement for shows 800 un-redeemed S.I.M. points. Call 08718738001 Identifier Code: 4955...


### 2.6 Flag the challenge subset (promotion words)
**Definition fixed on 22 November 2026, before any model was trained.**
A message belongs to the *challenge subset* if it contains at least one of these whole words, ignoring case:

`free, win, wins, won, winner, winning, offer, offers, prize, prizes`

These are the four words named in the approved proposal plus their common forms. "Whole word" means the word
is not part of a longer word: `FREE!` and `winner's` count, `freedom` and `won't` do not.

In [9]:
PROMO_WORDS = ["free", "win", "wins", "won", "winner", "winning", "offer", "offers", "prize", "prizes"]
PROMO_PATTERN = re.compile(r"(?<!\w)(?:" + "|".join(PROMO_WORDS) + r")(?!\w|['’]t\b)", flags=re.IGNORECASE)


def has_promo_word(text):
    """True if the message contains one of the challenge words as a whole word."""
    return PROMO_PATTERN.search(text) is not None


# Self-test of the rule on hand-made cases
assert has_promo_word("FREE entry!") and has_promo_word("You won a prize") and has_promo_word("the winner's list")
assert not has_promo_word("I won't be late") and not has_promo_word("I won’t") and not has_promo_word("freedom")

df["promo"] = df["text"].map(has_promo_word)
promo_name = df["promo"].map({True: "has promo word", False: "no promo word"})
pd.crosstab(df["label"], promo_name, margins=True, margins_name="Total")

promo,has promo word,no promo word,Total
label,,,
Not spam,73,4444,4517
Spam,283,358,641
Total,356,4802,5158


Which challenge words occur, by class (a message can contain more than one):

In [10]:
words = (df.assign(word=df["text"].map(lambda t: PROMO_PATTERN.findall(t.lower())))
          .explode("word").dropna(subset=["word"]))
word_table = pd.crosstab(words["word"], words["label"]).reindex(PROMO_WORDS, fill_value=0)
word_table.loc["(messages in subset)"] = df.loc[df["promo"], "label"].value_counts().reindex(word_table.columns)
word_table

label,Not spam,Spam
word,,
free,56,190
win,8,47
wins,2,1
won,0,56
winner,0,13
winning,3,0
offer,7,24
offers,1,9
prize,0,79


### 2.7 Split into training, development and test sets
* **70 / 15 / 15** split of template groups (Section 2.5), with seed 42.
* **Stratified by label × challenge flag**, so each split gets its fair share of the rare legitimate
  promotion-word messages.
* **Training** data fits the models, **development** data chooses their settings, and **test** data is used
  **once** for the final comparison.
* The split is saved to `data/splits.csv`. On later runs the notebook rebuilds the split and checks that it
  matches the saved file. The printed fingerprint lets anyone confirm they have the same split.

In [11]:
def split_by_group(data, seed=SEED):
    """Assign every message to train/dev/test by splitting template groups 70/15/15, stratified."""
    groups = (data.groupby("group_key", sort=False)
                  .agg(label=("label", "first"), promo=("promo", "max"), first_id=("msg_id", "min"))
                  .sort_values("first_id")
                  .reset_index())
    groups["stratum"] = groups["label"] + " | " + groups["promo"].map({True: "promo", False: "other"})
    train_g, temp_g = train_test_split(groups, test_size=0.30, stratify=groups["stratum"], random_state=seed)
    dev_g, test_g = train_test_split(temp_g, test_size=0.50, stratify=temp_g["stratum"], random_state=seed)
    assignment = {**dict.fromkeys(train_g["group_key"], "train"),
                  **dict.fromkeys(dev_g["group_key"], "dev"),
                  **dict.fromkeys(test_g["group_key"], "test")}
    return data["group_key"].map(assignment)


def split_fingerprint(table):
    """Short hash of the (msg_id, split) assignment, independent of operating system."""
    text = "\n".join(f"{i},{s}" for i, s in zip(table["msg_id"], table["split"]))
    return hashlib.sha256(text.encode()).hexdigest()[:12]


df["split"] = split_by_group(df)

# Leakage checks: every message assigned; no template group or exact text in more than one split
assert df["split"].notna().all()
assert (df.groupby("group_key")["split"].nunique() == 1).all()
assert (df.groupby("dup_key")["split"].nunique() == 1).all()

SPLITS_PATH = DATA_DIR / "splits.csv"
split_table = df[["msg_id", "label", "promo", "split"]]
if SPLITS_PATH.exists():
    saved = pd.read_csv(SPLITS_PATH)
    assert split_fingerprint(saved) == split_fingerprint(split_table), "Split differs from data/splits.csv"
    print("Split matches the saved file", SPLITS_PATH)
else:
    split_table.to_csv(SPLITS_PATH, index=False, lineterminator="\n")
    print("Saved the split to", SPLITS_PATH)
print("Split fingerprint:", split_fingerprint(split_table))

Saved the split to data/splits.csv
Split fingerprint: a8889b52ce6f


In [12]:
SPLITS = ["train", "dev", "test"]
split_counts = pd.crosstab(df["split"], [df["label"], df["promo"].map({True: "promo word", False: "other"})])
split_counts = split_counts.loc[SPLITS]
split_counts[("Total", "")] = split_counts.sum(axis=1)
split_counts.loc["Total"] = split_counts.sum()
save_result(split_counts, "split_counts")
print("Share of messages per split:",
      ", ".join(f"{s} {p:.1%}" for s, p in df["split"].value_counts(normalize=True).loc[SPLITS].items()))
split_counts

results/split_counts.csv: saved
Share of messages per split: train 69.8%, dev 15.0%, test 15.2%


label Not spam             Spam            Total
promo    other promo word other promo word      
split                                           
train     3111         51   244        194  3600
dev        668         11    50         44   773
test       665         11    64         45   785
Total     4444         73   358        283  5158

In [13]:
train_df = df[df["split"] == "train"].reset_index(drop=True)
dev_df = df[df["split"] == "dev"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)   # not looked at again until Section 5
print(f"train {len(train_df):,} | dev {len(dev_df):,} | test {len(test_df):,}")

train 3,600 | dev 773 | test 785


### 2.8 What the challenge looks like (training examples only)
To understand the difficulty without looking at test data, we show challenge messages from the **training set** only.

In [14]:
challenge_train = train_df[train_df["promo"]]
challenge_train.groupby("label").sample(5, random_state=SEED)[["label", "text"]]

,label,text
3194,Not spam,Once free call me sir.
2911,Not spam,"Lemme know when I can swing by and pick up, I'm free basically any time after 1 all this semester"
3333,Not spam,K..then come wenever u lik to come and also tel vikky to come by getting free time..:-)
1016,Not spam,Total video converter free download type this in google search:)
2299,Not spam,I will reach office around <DECIMAL> . & my mobile have problem. You cann't get my voice. So call you asa i'll free
3063,Spam,"Had your mobile 10 mths? Update to the latest Camera/Video phones for FREE. KEEP UR SAME NUMBER, Get extra free mins..."
3454,Spam,"u r subscribed 2 TEXTCOMP 250 wkly comp. 1st wk?s free question follows, subsequent wks charged@150p/msg.2 unsubscri..."
1529,Spam,<Forwarded from 88877>FREE entry into our £250 weekly comp just send the word ENTER to 88877 NOW. 18 T&C www.textcom...
432,Spam,New TEXTBUDDY Chat 2 horny guys in ur area 4 just 25p Free 2 receive Search postcode or at gaytextbuddy.com. TXT ONE...
92,Spam,Sunshine Quiz Wkly Q! Win a top Sony DVD player if u know which country the Algarve is in? Txt ansr to 82277. £1.50 ...


### 2.9 Evaluation plan (fixed before training)
**Recorded on 22 November 2026, before any model was trained and before the test set was examined.**

1. Both methods learn from the **training set only**. Every setting is chosen by **macro F1 on the development set**.
2. The **test set is used once**, to compare the two selected models on identical messages.
3. **Overall performance:** accuracy and macro F1 on the full test set.
4. **Answering the question ("more reliably"):** on the test messages in the challenge subset we report
   * **macro F1** (primary measure; the method with the higher value answers the question),
   * **false-alarm rate:** the share of legitimate promotion-word messages wrongly labelled Spam,
   * **spam recall:** the share of promotion-word spam that is caught.
5. The challenge subset contains only a small number of legitimate messages, so we report raw counts next to
   every percentage, examine **every** legitimate challenge message in the test set individually, and use
   McNemar's exact test to check whether the two methods' disagreements are more than chance.
6. The **Try it** cell uses whichever model has the higher development macro F1.

## 3. Method 1: TF-IDF word features + Logistic Regression
*Lead: Ali*

**From text to prediction.**
1. **TF-IDF** turns a message into a sparse vector with one entry per word (or word pair) in the training
   vocabulary. An entry is large when the word is frequent in this message but rare across training messages.
2. **Logistic regression** learns one weight per entry. The weighted sum becomes a probability of Spam, and the
   message is labelled Spam when that probability is above 0.5.

**Why this method.** It is the standard traditional baseline for short-text classification: fast, strong and easy
to inspect. Its weights show directly how much a single word such as "free" pushes a message towards Spam,
which is exactly the behaviour our challenge is about.

**Libraries and weights.** scikit-learn's `TfidfVectorizer` and `LogisticRegression`. No pretrained weights:
everything is learned from our training set.

**Fixed choices.**
* Lowercasing.
* Single-character tokens are kept: `u`, `r`, `2` and `4` are common in SMS, and scikit-learn's default
  tokenizer would drop them.
* Sublinear term frequency: a word repeated three times counts less than three times as much.

**Chosen on the development set.** Word n-grams (single words, or single words plus pairs), minimum document
frequency (1 or 2), regularisation strength `C` (0.1, 1, 10, 100, 1000; larger means weaker regularisation) and class weighting
(none, or *balanced*, which gives the rarer Spam class more weight). That makes 40 combinations, each trained on the training set and
scored by macro F1 on the development set.

### 3.1 Tuning helper (shared with Method 2)

In [15]:
def tune_on_dev(make_model, grid, X_train, y_train, X_dev, y_dev):
    """Train one model per setting on the training set and score it on the development set.
    Returns (table of all settings sorted by dev macro F1, best settings).
    Ties go to the setting that comes first in the grid."""
    results = []
    for params in ParameterGrid(grid):
        pred = make_model(**params).fit(X_train, y_train).predict(X_dev)
        results.append((params, accuracy_score(y_dev, pred), f1_score(y_dev, pred, average="macro")))
    best_params = max(results, key=lambda r: r[2])[0]        # max() keeps the first of equal scores
    table = pd.DataFrame([{**{k: str(v) for k, v in p.items()}, "dev accuracy": acc, "dev macro F1": f1}
                          for p, acc, f1 in results])
    table = table.sort_values("dev macro F1", ascending=False, kind="stable").reset_index(drop=True)
    return table, best_params


y_train = train_df["y"].to_numpy()
y_dev = dev_df["y"].to_numpy()

### 3.2 Tune on the development set, then train the final model

In [16]:
def make_method1(ngram_range=(1, 1), min_df=1, C=1.0, class_weight=None):
    """TF-IDF word features followed by logistic regression, as one scikit-learn pipeline."""
    return Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", sublinear_tf=True,
                                  ngram_range=ngram_range, min_df=min_df)),
        ("clf", LogisticRegression(C=C, class_weight=class_weight, max_iter=5000)),
    ])


M1_GRID = {
    "ngram_range": [(1, 1), (1, 2)],
    "min_df": [1, 2],
    "C": [0.1, 1, 10, 100, 1000],
    "class_weight": [None, "balanced"],
}

m1_table, m1_best = tune_on_dev(make_method1, M1_GRID, train_df["text"], y_train, dev_df["text"], y_dev)
save_result(m1_table, "method1_dev_grid")
print("Selected settings:", m1_best)
m1_table.head(8)

results/method1_dev_grid.csv: saved
Selected settings: {'C': 100, 'class_weight': 'balanced', 'min_df': 1, 'ngram_range': (1, 2)}


,C,class_weight,min_df,ngram_range,dev accuracy,dev macro F1
0,100,balanced,1,"(1, 2)",0.988357,0.972369
1,1000,balanced,2,"(1, 2)",0.988357,0.972369
2,1000,balanced,1,"(1, 2)",0.987063,0.969442
3,10,balanced,1,"(1, 2)",0.987063,0.969155
4,100,balanced,2,"(1, 2)",0.987063,0.969155
5,1000,balanced,1,"(1, 1)",0.987063,0.969155
6,1000,None,1,"(1, 2)",0.987063,0.968862
7,100,None,2,"(1, 2)",0.987063,0.968562


In [17]:
method1 = make_method1(**m1_best).fit(train_df["text"], y_train)    # final Method 1: training set only
print(f"Vocabulary: {len(method1.named_steps['tfidf'].vocabulary_):,} features")

Vocabulary: 39,221 features


### 3.3 What Method 1 learned
Large positive weights push a message towards Spam; large negative weights push it towards Not spam.
The second table shows the weight of each challenge word and its rank among all features. A highly ranked
"free" means the word on its own pushes legitimate messages towards Spam. A blank row means the word is not
in the training vocabulary.

In [18]:
features = method1.named_steps["tfidf"].get_feature_names_out()
weights = pd.Series(method1.named_steps["clf"].coef_[0], index=features)

TOP_N = 15
spam_side, ham_side = weights.nlargest(TOP_N), weights.nsmallest(TOP_N)
pd.DataFrame({
    "towards Spam": spam_side.index, "Spam weight": spam_side.round(2).to_numpy(),
    "towards Not spam": ham_side.index, "Not spam weight": ham_side.round(2).to_numpy(),
}, index=range(1, TOP_N + 1))

,towards Spam,Spam weight,towards Not spam,Not spam weight
1,txt,14.44,i,-14.89
2,call,13.02,me,-9.15
3,text,11.96,my,-8.48
4,free,11.85,ok,-5.62
5,stop,10.60,it,-5.52
6,mobile,10.34,so,-5.18
7,reply,9.75,what,-5.11
8,2,9.74,but,-4.74
9,www,9.65,in,-4.46
10,1,9.49,come,-4.32


In [19]:
promo_weights = pd.DataFrame({
    "weight": weights.reindex(PROMO_WORDS).round(2),
    "rank (1 = most spam-like)": weights.rank(ascending=False, method="min").reindex(PROMO_WORDS).astype("Int64"),
})
print(f"Ranks are out of {len(weights):,} features")
promo_weights

Ranks are out of 39,221 features


,weight,rank (1 = most spam-like)
free,11.85,4
win,5.46,42
wins,0.23,6760
won,6.42,25
winner,2.77,167
winning,-0.26,36436
offer,3.30,105
offers,0.95,1662
prize,8.27,15
prizes,NaN,<NA>


## 4. Method 2: Pretrained sentence embeddings + Logistic Regression
*Lead: Osama*

**From text to prediction.**
1. A pretrained sentence encoder, **all-MiniLM-L6-v2**, turns the whole message into a vector of 384 numbers
   (an *embedding*). It was trained so that sentences with similar meaning get nearby vectors, even when they
   use different words.
2. The encoder is **frozen**: its weights are not changed. Only a **logistic regression** on top of the
   embeddings is trained, on our training set.

**Why this method.** It tests whether a meaning-based representation separates *"I'm free after 5"* from
*"FREE entry to win £1000"* better than word counts do. It uses the **same classifier** as Method 1, so any
difference comes from the text representation, not from the classifier. MiniLM is small enough to run on a CPU.

**Libraries and weights.** The `sentence-transformers` library (Reimers & Gurevych, 2019) and the pretrained
`sentence-transformers/all-MiniLM-L6-v2` weights from Hugging Face, a MiniLM model (Wang et al., 2020). The
download is pinned to one revision of the model so results stay reproducible. The model's tokenizer lowercases
text itself, so the input is the same cleaned text that Method 1 uses.

**Chosen on the development set.** `C` (0.1 to 100,000) and class weighting (none or balanced): 14 combinations,
scored by dev macro F1. The `C` range reaches higher than in Method 1 because normalised embeddings have small
values, so weaker regularisation (larger `C`) can suit them. We widened the range until the best value was no
longer at its edge.

### 4.1 Load the encoder and embed the training and development messages
The first run downloads the model (about 90 MB). The test messages are embedded only in Section 5.

In [20]:
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"   # pinned commit of the model on Hugging Face

# CPU is fast enough for about 5,000 short messages and avoids GPU-specific numerical differences
encoder = SentenceTransformer(EMBED_MODEL, revision=EMBED_REVISION, device="cpu")
print("Encoder loaded | max tokens per message:", encoder.max_seq_length)


def embed(texts):
    """Encode messages as L2-normalised sentence embeddings, one row per message."""
    return encoder.encode(list(texts), batch_size=64, normalize_embeddings=True,
                          show_progress_bar=False, convert_to_numpy=True)


X_train_emb = embed(train_df["text"])
X_dev_emb = embed(dev_df["text"])
print("train", X_train_emb.shape, "| dev", X_dev_emb.shape)

Encoder loaded | max tokens per message: 256
train (3600, 384) | dev (773, 384)


### 4.2 Tune on the development set, then train the final model

In [21]:
def make_method2(C=1.0, class_weight=None):
    """Logistic regression on frozen sentence embeddings."""
    return LogisticRegression(C=C, class_weight=class_weight, max_iter=5000)


M2_GRID = {"C": [0.1, 1, 10, 100, 1000, 10000, 100000], "class_weight": [None, "balanced"]}

m2_table, m2_best = tune_on_dev(make_method2, M2_GRID, X_train_emb, y_train, X_dev_emb, y_dev)
save_result(m2_table, "method2_dev_grid")
print("Selected settings:", m2_best)
method2 = make_method2(**m2_best).fit(X_train_emb, y_train)       # final Method 2: training set only
m2_table

results/method2_dev_grid.csv: saved
Selected settings: {'C': 1000, 'class_weight': None}


,C,class_weight,dev accuracy,dev macro F1
0,1000,None,0.993532,0.984791
1,100,None,0.990944,0.978509
2,10,balanced,0.989651,0.975998
3,100,balanced,0.989651,0.975998
4,10,None,0.989651,0.975089
5,1000,balanced,0.988357,0.973119
6,10000,None,0.988357,0.972369
7,10000,balanced,0.987063,0.969997
8,100000,balanced,0.987063,0.969997
9,100000,None,0.987063,0.969442


### 4.3 Development-set comparison and the Try-it model
Both selected models on the **development** set. These settings were chosen on this set, so the numbers are
optimistic and are **not** the final results; those come from the test set in Section 5. The development
challenge subset is small, so treat those columns as an early indication only. As fixed in the evaluation plan
(2.9), the model with the higher dev macro F1 becomes the Try-it model.

In [22]:
METHOD_NAMES = {"m1": "Method 1: TF-IDF + LR", "m2": "Method 2: MiniLM embeddings + LR"}


def dev_row(pred):
    """Development-set scores for one method's predictions."""
    sub = dev_df["promo"].to_numpy()
    legit = (y_dev == 0) & sub
    return {"dev accuracy": accuracy_score(y_dev, pred),
            "dev macro F1": f1_score(y_dev, pred, average="macro"),
            "challenge accuracy": accuracy_score(y_dev[sub], pred[sub]),
            "challenge macro F1": f1_score(y_dev[sub], pred[sub], average="macro"),
            "legit challenge msgs flagged Spam": f"{int((pred[legit] == 1).sum())} of {int(legit.sum())}"}


dev_pred = {"m1": method1.predict(dev_df["text"]), "m2": method2.predict(X_dev_emb)}
dev_comparison = pd.DataFrame.from_dict({METHOD_NAMES[m]: dev_row(p) for m, p in dev_pred.items()}, orient="index")
save_result(dev_comparison, "dev_comparison")

TRY_IT_KEY = max(dev_pred, key=lambda m: dev_comparison.loc[METHOD_NAMES[m], "dev macro F1"])
print("Try-it model (higher dev macro F1):", METHOD_NAMES[TRY_IT_KEY])
dev_comparison.round(4)

results/dev_comparison.csv: saved
Try-it model (higher dev macro F1): Method 2: MiniLM embeddings + LR


,dev accuracy,dev macro F1,challenge accuracy,challenge macro F1,legit challenge msgs flagged Spam
Method 1: TF-IDF + LR,0.9884,0.9724,1.0000,1.0000,0 of 11
Method 2: MiniLM embeddings + LR,0.9935,0.9848,0.9818,0.9725,0 of 11


In [23]:
# Development-set probabilities, kept for later error analysis
dev_predictions = pd.DataFrame({
    "msg_id": dev_df["msg_id"], "y": y_dev, "promo": dev_df["promo"],
    "m1_spam_prob": method1.predict_proba(dev_df["text"])[:, 1],
    "m2_spam_prob": method2.predict_proba(X_dev_emb)[:, 1],
}).set_index("msg_id")
save_result(dev_predictions, "dev_predictions")

results/dev_predictions.csv: saved


### 4.4 Prediction function for Results and Try it
One function returns the probability of Spam from either trained model. Nothing is retrained when it is called.

In [24]:
def spam_probability(method, texts):
    """Probability of Spam for each text from the trained Method 1 ('m1') or Method 2 ('m2')."""
    texts = list(texts)
    if method == "m1":
        return method1.predict_proba(texts)[:, 1]
    return method2.predict_proba(embed(texts))[:, 1]


# Sanity check on the two examples given in the project brief
brief_examples = ["You won a cash prize. Pay a fee now to claim it", "Please bring your notebook to tomorrow's lab"]
pd.DataFrame({"message": brief_examples,
              **{f"{METHOD_NAMES[m]}: P(Spam)": spam_probability(m, brief_examples).round(3) for m in METHOD_NAMES}})

,message,Method 1: TF-IDF + LR: P(Spam),Method 2: MiniLM embeddings + LR: P(Spam)
0,You won a cash prize. Pay a fee now to claim it,0.999,0.991
1,Please bring your notebook to tomorrow's lab,0.155,0.000


## 5. Results
*Lead: Hussain*

The test set is used here for the first time, to evaluate the two models selected in Sections 3–4, following the
plan fixed in Section 2.9. **Nothing in Sections 2–4 was changed after these results were seen** (first
evaluated on 22 November 2026). Re-running the notebook re-evaluates the same frozen models; it does not
re-tune anything.

### 5.1 Embed the test messages and predict

In [25]:
X_test_emb = embed(test_df["text"])
y_test = test_df["y"].to_numpy()

test_pred = {"m1": method1.predict(test_df["text"]), "m2": method2.predict(X_test_emb)}
test_prob = {"m1": method1.predict_proba(test_df["text"])[:, 1], "m2": method2.predict_proba(X_test_emb)[:, 1]}

test_predictions = pd.DataFrame({
    "msg_id": test_df["msg_id"], "label": test_df["label"], "promo": test_df["promo"],
    "m1_spam_prob": test_prob["m1"], "m1_pred": test_pred["m1"],
    "m2_spam_prob": test_prob["m2"], "m2_pred": test_pred["m2"],
    "text": test_df["text"],
}).set_index("msg_id")
save_result(test_predictions, "test_predictions")
print(f"Test set: {len(test_df)} messages ({int(y_test.sum())} Spam, {int((y_test == 0).sum())} Not spam)")

results/test_predictions.csv: saved
Test set: 785 messages (109 Spam, 676 Not spam)


### 5.2 Overall performance on the full test set
Accuracy and macro F1 are the two measures required by the brief. Precision and recall for each class show
*where* each method makes its mistakes.

In [26]:
def overall_row(y, pred):
    """Overall test-set scores for one method."""
    return {"accuracy": accuracy_score(y, pred),
            "macro F1": f1_score(y, pred, average="macro"),
            "Spam precision": precision_score(y, pred, pos_label=1, zero_division=0),
            "Spam recall": recall_score(y, pred, pos_label=1, zero_division=0),
            "Not spam precision": precision_score(y, pred, pos_label=0, zero_division=0),
            "Not spam recall": recall_score(y, pred, pos_label=0, zero_division=0)}


test_overall = pd.DataFrame.from_dict(
    {METHOD_NAMES[m]: overall_row(y_test, p) for m, p in test_pred.items()}, orient="index")
save_result(test_overall, "test_overall")
test_overall.round(4)

results/test_overall.csv: saved


,accuracy,macro F1,Spam precision,Spam recall,Not spam precision,Not spam recall
Method 1: TF-IDF + LR,0.9796,0.9571,0.9346,0.9174,0.9867,0.9896
Method 2: MiniLM embeddings + LR,0.9847,0.9680,0.9450,0.9450,0.9911,0.9911


In [27]:
def confusion_table(y, pred):
    """Confusion matrix with readable row and column names."""
    return pd.DataFrame(confusion_matrix(y, pred, labels=[0, 1]),
                        index=["true Not spam", "true Spam"], columns=["predicted Not spam", "predicted Spam"])


test_confusion = pd.concat({METHOD_NAMES[m]: confusion_table(y_test, p) for m, p in test_pred.items()}, axis=1)
save_result(test_confusion, "test_confusion")
test_confusion

results/test_confusion.csv: saved


Method 1: TF-IDF + LR                 \
                 predicted Not spam predicted Spam   
true Not spam                   669              7   
true Spam                         9            100   

              Method 2: MiniLM embeddings + LR                 
                            predicted Not spam predicted Spam  
true Not spam                              670              6  
true Spam                                    6            103

### 5.3 The challenge subset: answering the experimental question
The measures fixed in Section 2.9, on the test messages that contain a promotion word:
* **macro F1** is the primary measure; the method with the higher value answers the question,
* **false-alarm rate** is the share of legitimate promotion-word messages wrongly labelled Spam,
* **spam recall** is the share of promotion-word spam that is caught.

Raw counts are shown next to every rate. For context, the last column gives macro F1 on the test messages
*without* promotion words.

In [28]:
sub = test_df["promo"].to_numpy()
legit = sub & (y_test == 0)          # legitimate messages with a promotion word
promo_spam = sub & (y_test == 1)     # spam messages with a promotion word
print(f"Challenge subset in the test set: {sub.sum()} messages ({legit.sum()} legitimate, {promo_spam.sum()} spam)")


def challenge_row(pred):
    """Challenge-subset measures for one method (Section 2.9)."""
    false_alarms = int((pred[legit] == 1).sum())
    caught = int((pred[promo_spam] == 1).sum())
    return {"challenge macro F1": f1_score(y_test[sub], pred[sub], average="macro"),
            "challenge accuracy": accuracy_score(y_test[sub], pred[sub]),
            "false alarms": false_alarms, "legit messages": int(legit.sum()),
            "false-alarm rate": false_alarms / legit.sum(),
            "spam caught": caught, "promo spam": int(promo_spam.sum()),
            "spam recall": caught / promo_spam.sum(),
            "other messages macro F1": f1_score(y_test[~sub], pred[~sub], average="macro")}


test_challenge = pd.DataFrame.from_dict({METHOD_NAMES[m]: challenge_row(p) for m, p in test_pred.items()},
                                        orient="index")
save_result(test_challenge, "test_challenge")
test_challenge.round(4)

Challenge subset in the test set: 56 messages (11 legitimate, 45 spam)
results/test_challenge.csv: saved


,challenge macro F1,challenge accuracy,false alarms,legit messages,false-alarm rate,spam caught,promo spam,spam recall,other messages macro F1
Method 1: TF-IDF + LR,0.9121,0.9464,2,11,0.1818,44,45,0.9778,0.9431
Method 2: MiniLM embeddings + LR,0.9707,0.9821,1,11,0.0909,45,45,1.0000,0.9526


### 5.4 Is the difference more than chance? McNemar's exact test
Both methods labelled the same messages, so we compare them message by message. We count the messages that
**only Method 1** labelled correctly and those that **only Method 2** labelled correctly. If the two methods were
equally good, a message one method got wrong would be equally likely to be either method's mistake. The exact
binomial test gives the probability (p-value) of a split at least this uneven under that assumption.
A p-value below 0.05 is usually read as a real difference. With few disagreements the test has little power, so
a large p-value means *no evidence of a difference*, not *no difference*.

In [29]:
def mcnemar_exact(y, pred_a, pred_b):
    """McNemar's exact test on paired predictions of two methods."""
    a_right, b_right = pred_a == y, pred_b == y
    only_a, only_b = int((a_right & ~b_right).sum()), int((~a_right & b_right).sum())
    p_value = binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0
    return {"only Method 1 correct": only_a, "only Method 2 correct": only_b, "exact p-value": p_value}


test_mcnemar = pd.DataFrame.from_dict({
    "full test set": mcnemar_exact(y_test, test_pred["m1"], test_pred["m2"]),
    "challenge subset": mcnemar_exact(y_test[sub], test_pred["m1"][sub], test_pred["m2"][sub]),
}, orient="index")
save_result(test_mcnemar, "test_mcnemar")
test_mcnemar.round(4)

results/test_mcnemar.csv: saved


,only Method 1 correct,only Method 2 correct,exact p-value
full test set,6,10,0.4545
challenge subset,0,2,0.5000


### 5.5 Every legitimate challenge message in the test set
These messages are the heart of the question, and there are few enough to inspect each one.

In [30]:
legit_table = pd.DataFrame({
    "promo words": test_df.loc[legit, "text"].map(lambda t: ", ".join(PROMO_PATTERN.findall(t))),
    "M1 P(Spam)": test_prob["m1"][legit].round(3),
    "M2 P(Spam)": test_prob["m2"][legit].round(3),
    "text": test_df.loc[legit, "text"],
}).set_index(test_df.loc[legit, "msg_id"])
save_result(legit_table, "test_legit_challenge")
legit_table

results/test_legit_challenge.csv: saved


,promo words,M1 P(Spam),M2 P(Spam),text
msg_id,,,,
181,offer,0.004,0.000,You lifted my hopes with the offer of money. I am in need. Especially when the end of the month approaches and it hu...
486,win,0.086,0.000,If we win its really no 1 side for long time.
495,free,0.263,0.025,Are you free now?can i call now?
501,"win, win",0.203,0.000,"Fighting with the world is easy, u either win or lose bt fightng with some1 who is close to u is dificult if u lose ..."
744,free,0.180,0.000,Ok. There may be a free gym about.
1759,free,0.003,0.000,Sorry i'm not free...
2289,offer,0.907,0.723,Dont you have message offer
3333,free,0.001,0.000,Ok... The theory test? when are ü going to book? I think it's on 21 may. Coz thought wanna go out with jiayin. But s...
4009,free,0.010,0.000,"Forgot you were working today! Wanna chat, but things are ok so drop me a text when you're free / bored etc and i'll..."


### 5.6 Error analysis
Every test message that **at least one** method got wrong, with evidence for *why*:
* **Method 1 evidence:** the words in the message that pushed the prediction most towards Spam and towards Not
  spam (TF-IDF value × learned weight; the model's constant intercept term is not shown).
* **Method 2 evidence:** the three training messages closest in meaning (cosine similarity of embeddings), with
  their labels. A message whose nearest neighbours carry the other label is likely to be misclassified.

In [31]:
coef = weights.to_numpy()                      # Method 1 weights, aligned with `features` (Section 3.3)


def m1_top_terms(text, k=4):
    """The k terms in a message that push Method 1 most towards Spam, and most towards Not spam."""
    row = method1.named_steps["tfidf"].transform([text]).tocsr()
    contrib = row.data * coef[row.indices]
    order = np.argsort(contrib)
    to_spam = [f"{features[row.indices[j]]} ({contrib[j]:+.1f})" for j in order[::-1] if contrib[j] > 0][:k]
    to_ham = [f"{features[row.indices[j]]} ({contrib[j]:+.1f})" for j in order if contrib[j] < 0][:k]
    return ", ".join(to_spam), ", ".join(to_ham)


def m2_neighbours(vector, k=3):
    """The k training messages whose embeddings are closest to `vector`, with label and similarity."""
    sims = X_train_emb @ vector
    top = np.argsort(-sims)[:k]
    return " | ".join(f"[{train_df['label'].iat[i]} {sims[i]:.2f}] {train_df['text'].iat[i][:70]}" for i in top)


wrong_any = np.flatnonzero((test_pred["m1"] != y_test) | (test_pred["m2"] != y_test))
error_rows = []
for i in wrong_any:
    to_spam, to_ham = m1_top_terms(test_df["text"].iat[i])
    error_rows.append({
        "msg_id": test_df["msg_id"].iat[i], "true label": test_df["label"].iat[i],
        "promo": bool(test_df["promo"].iat[i]),
        "M1": "wrong" if test_pred["m1"][i] != y_test[i] else "right", "M1 P(Spam)": round(test_prob["m1"][i], 3),
        "M2": "wrong" if test_pred["m2"][i] != y_test[i] else "right", "M2 P(Spam)": round(test_prob["m2"][i], 3),
        "text": test_df["text"].iat[i],
        "M1 terms towards Spam": to_spam, "M1 terms towards Not spam": to_ham,
        "M2 nearest training messages": m2_neighbours(X_test_emb[i]),
    })
test_errors = pd.DataFrame(error_rows).set_index("msg_id")
save_result(test_errors, "test_errors")
print(f"Method 1 errors: {(test_pred['m1'] != y_test).sum()} | Method 2 errors: {(test_pred['m2'] != y_test).sum()} "
      f"| messages wrong for at least one method: {len(test_errors)}")
test_errors[["true label", "promo", "M1", "M1 P(Spam)", "M2", "M2 P(Spam)", "text"]]

results/test_errors.csv: saved
Method 1 errors: 16 | Method 2 errors: 12 | messages wrong for at least one method: 22


,true label,promo,M1,M1 P(Spam),M2,M2 P(Spam),text
msg_id,,,,,,,
164,Spam,False,right,0.999,wrong,0.000,-PLS STOP bootydelious (32/F) is inviting you to be her friend. Reply YES-434 or NO-434 See her: www.SMS.ac/u/bootyd...
574,Not spam,False,wrong,0.975,right,0.000,Waiting for your call.
700,Not spam,False,wrong,0.763,right,0.000,K..u also dont msg or reply to his msg..
1082,Not spam,False,wrong,0.610,right,0.000,Can u get pic msgs to your phone?
1430,Spam,False,wrong,0.057,wrong,0.003,For sale - arsenal dartboard. Good condition but no doubles or trebles!
1893,Spam,False,wrong,0.353,right,1.000,CALL 09090900040 & LISTEN TO EXTREME DIRTY LIVE CHAT GOING ON IN THE OFFICE RIGHT NOW TOTAL PRIVACY NO ONE KNOWS YOU...
2246,Not spam,False,right,0.027,wrong,0.821,How did you find out in a way that didn't include all of these details
2289,Not spam,True,wrong,0.907,wrong,0.723,Dont you have message offer
2364,Spam,False,wrong,0.239,right,0.993,Fantasy Football is back on your TV. Go to Sky Gamestar on Sky Active and play £250k Dream Team. Scoring starts on S...


In [32]:
# Evidence for each error (Method 1 terms and Method 2 nearest neighbours)
test_errors[["M1 terms towards Spam", "M1 terms towards Not spam", "M2 nearest training messages"]]

,M1 terms towards Spam,M1 terms towards Not spam,M2 nearest training messages
msg_id,,,
164,"stop (+2.4), www (+1.1), reply (+1.0), to (+0.7)","her (-0.2), u (-0.2), yes (-0.2), pls (-0.2)",[Spam 0.59] Natalja (25/F) is inviting you to be her friend. Reply YES-440 or NO-4 | [Spam 0.52] Natalie (20/F) is i...
574,"call (+3.3), your (+1.9), for (+1.5), for your (+1.2)",your call (-0.9),"[Not spam 0.78] I am waiting for your call sir. | [Not spam 0.77] Sir, i am waiting for your call. | [Not spam 0.63]..."
700,"msg (+2.4), reply (+2.3), or (+1.3), to (+0.9)","k (-0.9), u (-0.4), dont (-0.4), his (-0.4)",[Not spam 0.66] Don know. I did't msg him recently. | [Not spam 0.63] Ok... I din get ur msg... | [Not spam 0.53] Ev...
1082,"your (+1.4), phone (+1.1), to (+1.0), can u (+0.8)","can (-0.7), u (-0.5), u get (-0.1)","[Spam 0.56] Hottest pics straight to your phone!! See me getting Wet and Wanting, | [Spam 0.54] Do you want a New N..."
1430,"or (+1.4), for (+1.0), arsenal (+0.5), sale (+0.1)","but (-0.9), good (-0.5), no (-0.2), but no (-0.0)",[Not spam 0.43] Ok i found dis pierre cardin one which looks normal costs 20 its on sa | [Not spam 0.43] It's é only...
1893,"call (+1.3), chat (+1.0), your (+0.7), min (+0.7)","going (-0.4), in (-0.4), right (-0.4), one (-0.3)",[Spam 0.54] I'd like to tell you my deepest darkest fantasies. Call me 09094646631 | [Spam 0.53] Hack Chat. Get back...
2246,"these (+0.7), find out (+0.6), all of (+0.5), of these (+0.5)","how (-0.6), in (-0.4), that (-0.4), way (-0.4)","[Not spam 0.34] I dont have any of your file in my bag..i was in work when you called | [Not spam 0.33] <#> , that'..."
2289,"you have (+3.0), message (+2.4), offer (+1.8), have (+0.5)","dont (-0.7), you (-0.5)",[Not spam 0.51] Ok lor ü reaching then message me. | [Not spam 0.50] did u get that message | [Not spam 0.48] Oh yes...
2364,"to (+0.8), your (+0.7), now (+0.6), play (+0.4)","so (-0.5), go (-0.1), is back (-0.1), team (-0.1)",[Spam 0.49] 87077: Kick off a new season with 2wks FREE goals & news to ur mobile! | [Spam 0.48] YOUR CHANCE TO BE O...


### 5.7 Explaining the errors
Method 1 made 16 errors (7 false alarms, 9 missed spam) and Method 2 made 12 (6 and 6). Six messages fooled
both. The explanations below use the evidence in 5.6; message numbers are `msg_id`s.

**1. Short legitimate messages dominated by one spam-associated word (Method 1 false alarms).**
Five of Method 1's seven false alarms are messages of 4–9 words. *"Waiting for your call."* (574) got P(Spam) =
0.975 because `call` (+3.3), `your` (+1.9) and `for your` (+1.2) are strong spam features, and nothing in so
short a message pulls the other way. *"Just send a text. We'll skype later."* (5164) fails the same way through
`text` (+2.8). Method 2 labels both correctly: their nearest training messages are near-identical legitimate
texts, such as *"I am waiting for your call sir."* (similarity 0.78). This is the same mechanism as our challenge,
with ordinary words such as `call` and `text` playing the role of `free`.

**2. Legitimate challenge messages (5.5).** *"Dont you have message offer"* (2289) fooled **both** methods.
It is five words long, with no context, and the phrase reads like a carrier promotion. Method 1 is pushed by
`you have` (+3.0), `message` (+2.4) and `offer` (+1.8). Message 4488 is a code-mixed (Bengali–English)
message ending in *"...great phone receiving quality wins."* Method 1 flagged it (0.663), but mostly because of
`call` (+3.7), which appears three times in *"miss call"*. The challenge word `wins` itself has a near-zero
weight. Method 2 was confident it is legitimate (0.000). Its nearest neighbours are other messages about missed
calls.

**3. Promotion-word spam missed by Method 1.** Message 5542 is an all-capitals spam message written like a
customer complaint (*"ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE MINS..."*). `free` pushes it towards Spam
(+1.5), but words typical of personal messages (`i`, `in`, `got`, `if`) outweigh it, giving 0.053.
Method 2 places it next to capitalised spam with prize amounts and is certain it is spam (1.000).

**4. Spam that does not look like spam (both methods wrong).** Three missed spam messages contain no
promotional cue at all: a quotation about teenagers (2774), a horoscope-style text (2823) and a classified ad for
a dartboard (1430). They appear to come from subscription services or advertisers, so they count as spam because
of *who sent them*, which the text alone cannot reveal. Neither words nor meaning help here. This limitation
applies to any text-only filter. Both methods also missed an Interflora flower advert written in plain, polite
sentences (2821).

**5. Unusual conversational messages (Method 2 false alarms).** Method 2 flagged ordinary messages such as
2246 (*"How did you find out in a way that didn't include all of these details"*, 0.821) and 4116, a long
message about casting. These have no close neighbour in the training data (highest similarity 0.34), so the
classifier is extrapolating. Method 1 gets them right because they contain no spam-associated words.

**6. Limits of the evidence.** Method 2 missed spam message 164 with P(Spam) = 0.000, even though its three
nearest training messages are all spam *"friend invitation"* messages. A linear classifier on embeddings is not
a vote among neighbours: with weak regularisation (`C = 1000`) it can place a message confidently on the wrong
side of its boundary. Separately, Method 1's false alarm on message 3306, a chain message in transliterated
Kannada, is driven by the digits `1`, `2` and `4`. That is a side effect of our choice to keep single-character
tokens (Section 3).

### 5.8 Answer to the experimental question
*Which method distinguishes legitimate promotion-word SMS messages from genuine spam more reliably?*

On the test set, **Method 2 (MiniLM embeddings + LR) handled the challenge subset better**:
* challenge macro F1 **0.971 vs 0.912**,
* **1 of 11** legitimate promotion-word messages wrongly flagged, against **2 of 11**,
* **45 of 45** promotion-word spam messages caught, against **44 of 45**.

It was also better on the full test set: accuracy 0.985 vs 0.980, macro F1 0.968 vs 0.957.

**The evidence is weak, and we state it as such.** On the challenge subset the methods disagree on only two
messages (4488 and 5542), both in Method 2's favour, and McNemar's exact test gives p = 0.50. On the full test set
the split is 10 messages to 6 in Method 2's favour, p = 0.45. On the development set the challenge-subset order
was reversed: Method 1 scored perfectly there, while Method 2 missed one promotion-word spam message. So the
results are *consistent with* the idea that meaning-based representations are less misled by spam-associated
words, as cases 1 and 2 in 5.7 illustrate. They do not *establish* it: a much larger set of legitimate
promotion-word messages would be needed.

**What we did learn with more confidence.** Method 1's false alarms are driven less by our four promotion words
than by other spam-associated words (`call`, `text`, `msg`, `you have`) in very short messages. Both methods
share a ceiling: spam that is only spam because of its sender (case 4).

## 6. Try it
*Lead: Hussain*

Classify any new SMS message with the trained models. **Nothing is retrained**: the function below only calls the
models trained in Sections 3–4. Each message gets the same cleaning as the training data (Section 2.3). The
prediction comes from the **Try-it model**, chosen by development macro F1 in Section 4.3. The other model's
opinion is shown for comparison, which makes disagreements on promotion-word messages easy to see in a demo.

### 6.1 The classification function

In [33]:
def to_label(p_spam):
    """Turn a probability of Spam into a label (same 0.5 threshold the models use)."""
    return "Spam" if p_spam > 0.5 else "Not spam"


def classify_sms(message, show_other_model=True):
    """Classify one SMS message with the trained Try-it model, without retraining.
    Prints a readable result and returns it as a dict, or prints a hint and returns None for empty or invalid input."""
    if not isinstance(message, str):
        print('No prediction: please give the message as text, for example classify_sms("See you at 5").')
        return None
    text = clean_text(message)                    # same cleaning as the training data
    if not text:
        print("No prediction: the message is empty. Please type an SMS message and run the cell again.")
        return None

    p_spam = {m: float(spam_probability(m, [text])[0]) for m in METHOD_NAMES}
    promo_found = PROMO_PATTERN.findall(text)
    other = "m1" if TRY_IT_KEY == "m2" else "m2"

    print(f'Message     : "{text}"')
    print(f"Prediction  : {to_label(p_spam[TRY_IT_KEY]).upper():<9} P(Spam) = {p_spam[TRY_IT_KEY]:.3f}   "
          f"[{METHOD_NAMES[TRY_IT_KEY]}]")
    if show_other_model:
        print(f"Other model : {to_label(p_spam[other]):<9} P(Spam) = {p_spam[other]:.3f}   [{METHOD_NAMES[other]}]")
    if promo_found:
        print(f"Note        : contains the promotion word(s) {', '.join(promo_found)}, "
              "the kind of message this project studies")
    return {"text": text, "prediction": to_label(p_spam[TRY_IT_KEY]), "p_spam": p_spam[TRY_IT_KEY],
            "model": METHOD_NAMES[TRY_IT_KEY], "other_model_p_spam": p_spam[other], "promo_words": promo_found}

### 6.2 Try your own message
Type a message between the quotes and run the cell. To be prompted for a message instead, set
`ASK_FOR_INPUT = True`. It is off by default so that *Run all* never stops to wait for typing.

In [34]:
ASK_FOR_INPUT = False                                          # True = type the message when the cell runs
MESSAGE = "You won a cash prize. Pay a fee now to claim it"     # or edit this text

if "method2" not in globals():
    print("The models are not loaded yet: choose Runtime → Run all (or run every cell above) first.")
else:
    if ASK_FOR_INPUT:
        MESSAGE = input("Type an SMS message: ")
    result = classify_sms(MESSAGE)

Message     : "You won a cash prize. Pay a fee now to claim it"
Prediction  : SPAM      P(Spam) = 0.991   [Method 2: MiniLM embeddings + LR]
Other model : Spam      P(Spam) = 0.999   [Method 1: TF-IDF + LR]
Note        : contains the promotion word(s) won, prize, the kind of message this project studies


### 6.3 More examples of new input
The two example messages from the project brief, followed by inputs that cannot be classified (empty text,
spaces only, and a value that is not text), to show how they are handled.

In [35]:
demo_inputs = [
    "You won a cash prize. Pay a fee now to claim it",   # brief's spam example
    "Please bring your notebook to tomorrow's lab",      # brief's not-spam example
    "",                                                  # empty input
    "     ",                                             # spaces only
    None,                                                # not text
]
for demo in demo_inputs:
    print("-" * 100)
    classify_sms(demo)

----------------------------------------------------------------------------------------------------
Message     : "You won a cash prize. Pay a fee now to claim it"
Prediction  : SPAM      P(Spam) = 0.991   [Method 2: MiniLM embeddings + LR]
Other model : Spam      P(Spam) = 0.999   [Method 1: TF-IDF + LR]
Note        : contains the promotion word(s) won, prize, the kind of message this project studies
----------------------------------------------------------------------------------------------------
Message     : "Please bring your notebook to tomorrow's lab"
Prediction  : NOT SPAM  P(Spam) = 0.000   [Method 2: MiniLM embeddings + LR]
Other model : Not spam  P(Spam) = 0.155   [Method 1: TF-IDF + LR]
----------------------------------------------------------------------------------------------------
No prediction: the message is empty. Please type an SMS message and run the cell again.
----------------------------------------------------------------------------------------------------

### 6.4 Our original labelled examples
Each team member writes **two messages of their own** (not copied from the dataset), with the true label and
the date written. They are kept out of training, development and test data and are reported **separately from
the test scores**. Including some legitimate messages that use words such as "free" or "win" makes them a
useful probe of our challenge. The check below warns if a message already appears in the dataset.

In [36]:
# Two messages written by each team member (not taken from the dataset)
ORIGINAL_EXAMPLES = [
    # Ali: one hard legitimate message, one obvious spam
    {"author": "Ali",     "date": "2026-11-23", "label": "Not spam", "text": "Are you free after class? Let's study together for the NLP quiz"},
    {"author": "Ali",     "date": "2026-11-23", "label": "Spam",     "text": "Congratulations! You have won a FREE iPhone. Reply YES now to claim your prize"},

    # Osama: legitimate message with "win", spam with "offer"
    {"author": "Osama",   "date": "2026-11-23", "label": "Not spam", "text": "I hope our team will win the match tonight, see you at the stadium"},
    {"author": "Osama",   "date": "2026-11-23", "label": "Spam",     "text": "Special offer just for you! Get 50% discount, call 0512345678 before midnight"},

    # Hussain: legitimate message with "offer", plain normal message
    {"author": "Hussain", "date": "2026-11-23", "label": "Not spam", "text": "My uncle got a job offer in Riyadh, we will celebrate at home tomorrow"},
    {"author": "Hussain", "date": "2026-11-23", "label": "Not spam", "text": "Please send me the lecture slides when you get home"},
]
print(f"{len(ORIGINAL_EXAMPLES)} of 6 original examples added so far")

6 of 6 original examples added so far


In [37]:
def check_original_examples(examples):
    """Validate the team's original examples and classify them with both models (no retraining)."""
    if not examples:
        print("No original examples yet: each member should add two messages in the cell above.")
        return None
    dataset_texts = set(raw_df["text_raw"].map(clean_text).str.lower())
    rows = []
    for ex in examples:
        text = clean_text(ex.get("text", ""))
        assert text, f"Empty text in example: {ex}"
        assert ex.get("label") in ("Spam", "Not spam"), f"Label must be 'Spam' or 'Not spam': {ex}"
        if text.lower() in dataset_texts:
            print(f"Warning: {ex.get('author')}'s example is already in the dataset, so it is not original: {text}")
        p_spam = {m: float(spam_probability(m, [text])[0]) for m in METHOD_NAMES}
        rows.append({"author": ex.get("author"), "date": ex.get("date"), "true label": ex["label"],
                     "promo word": bool(PROMO_PATTERN.search(text)), "text": text,
                     **{f"{METHOD_NAMES[m]}": f"{to_label(p)} ({p:.3f})" for m, p in p_spam.items()},
                     **{f"{m} correct": to_label(p) == ex["label"] for m, p in p_spam.items()}})
    table = pd.DataFrame(rows)
    for m in METHOD_NAMES:
        print(f"{METHOD_NAMES[m]}: {int(table[f'{m} correct'].sum())} of {len(table)} correct")
    save_result(table, "original_examples")
    return table


check_original_examples(ORIGINAL_EXAMPLES)

Method 1: TF-IDF + LR: 6 of 6 correct
Method 2: MiniLM embeddings + LR: 6 of 6 correct
results/original_examples.csv: saved


,author,date,true label,promo word,text,Method 1: TF-IDF + LR,Method 2: MiniLM embeddings + LR,m1 correct,m2 correct
0,Ali,2026-11-23,Not spam,True,Are you free after class? Let's study together for the NLP quiz,Not spam (0.126),Not spam (0.000),True,True
1,Ali,2026-11-23,Spam,True,Congratulations! You have won a FREE iPhone. Reply YES now to claim your prize,Spam (1.000),Spam (0.999),True,True
2,Osama,2026-11-23,Not spam,True,"I hope our team will win the match tonight, see you at the stadium",Not spam (0.006),Not spam (0.000),True,True
3,Osama,2026-11-23,Spam,True,"Special offer just for you! Get 50% discount, call 0512345678 before midnight",Spam (0.772),Spam (1.000),True,True
4,Hussain,2026-11-23,Not spam,True,"My uncle got a job offer in Riyadh, we will celebrate at home tomorrow",Not spam (0.001),Not spam (0.000),True,True
5,Hussain,2026-11-23,Not spam,False,Please send me the lecture slides when you get home,Not spam (0.001),Not spam (0.000),True,True


## 7. Sources

**Data**
* T. A. Almeida and J. M. Gómez Hidalgo, "SMS Spam Collection," UCI Machine Learning Repository, 2011.
  doi:10.24432/C5CC84. Corpus readme: free use with citation; copyright held by the authors.
* T. A. Almeida, J. M. Gómez Hidalgo and A. Yamakami, "Contributions to the study of SMS spam filtering:
  new collection and results," in *Proc. ACM DocEng*, 2011. doi:10.1145/2034691.2034742.

**Libraries**
* pandas, NumPy, SciPy, PyTorch, scikit-learn (F. Pedregosa et al., "Scikit-learn: Machine learning in Python,"
  *JMLR*, vol. 12, 2011), sentence-transformers.

**Statistical test**
* Q. McNemar, "Note on the sampling error of the difference between correlated proportions or percentages,"
  *Psychometrika*, vol. 12, no. 2, pp. 153–157, 1947.

**Pretrained model**
* `sentence-transformers/all-MiniLM-L6-v2`, Hugging Face model hub, revision pinned in Section 4.1.
* N. Reimers and I. Gurevych, "Sentence-BERT: Sentence embeddings using Siamese BERT-networks," in *Proc.
  EMNLP-IJCNLP*, 2019.
* W. Wang, F. Wei, L. Dong, H. Bao, N. Yang and M. Zhou, "MiniLM: Deep self-attention distillation for
  task-agnostic compression of pre-trained transformers," in *Proc. NeurIPS*, 2020.

**AI assistance**
* Parts of the data-preparation and model code and the notebook structure were drafted with help from Claude (Anthropic),
  an AI assistant. The team reviewed, ran and edited all code and is responsible for it.